# Arctic Vessel Analysis Demo

**Student Research Project**: Educational demonstration of machine learning techniques for Arctic maritime data analysis.

This notebook demonstrates:
- Simple vessel detection and tracking
- Pattern analysis of vessel behavior
- Anomaly detection using autoencoders
- Risk scoring methodology

**Academic Purpose**: This is an educational project for learning data science applications in maritime research.

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Vessel Detection and Data Processing

We'll start by simulating AIS data and demonstrating vessel detection capabilities.

In [ ]:
# Import our simple vessel detection module
from detection.simple_vessel_detection import VesselDetector, PatternDetector, create_sample_ais_data

# Create sample AIS data for demonstration
print("Generating sample AIS data for Arctic waters...")
ais_data = create_sample_ais_data(n_vessels=25, n_positions_per_vessel=20)
print(f"Generated {len(ais_data)} AIS messages from {len(set(msg['mmsi'] for msg in ais_data))} vessels")

# Initialize vessel detector
detector = VesselDetector(proximity_threshold_km=3.0)

# Process AIS data
print("\nProcessing vessel positions...")
vessel_df = detector.detect_vessels_from_ais(ais_data)
vessel_df = detector.calculate_vessel_features(vessel_df)

print(f"Processed {len(vessel_df)} vessel positions")
print(f"Unique vessels: {vessel_df['vessel_id'].nunique()}")
print(f"Time span: {vessel_df['timestamp'].min()} to {vessel_df['timestamp'].max()}")

In [ ]:
# Display sample of vessel data
print("Sample of processed vessel data:")
display_columns = ['vessel_id', 'latitude', 'longitude', 'speed', 'vessel_type', 'in_arctic', 'speed_category']
print(vessel_df[display_columns].head(10))

## 2. Vessel Distribution Visualization

Let's visualize the vessel positions and basic statistics.

In [ ]:
# Create visualization of vessel positions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Vessel positions on map
ax1 = axes[0, 0]
scatter = ax1.scatter(vessel_df['longitude'], vessel_df['latitude'], 
                     c=vessel_df['speed'], cmap='viridis', alpha=0.6, s=50)
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.set_title('Vessel Positions (colored by speed)')
plt.colorbar(scatter, ax=ax1, label='Speed (km/h)')
ax1.grid(True, alpha=0.3)

# 2. Speed distribution
ax2 = axes[0, 1]
ax2.hist(vessel_df['speed'], bins=20, alpha=0.7, edgecolor='black')
ax2.set_xlabel('Speed (km/h)')
ax2.set_ylabel('Number of Observations')
ax2.set_title('Speed Distribution')
ax2.axvline(vessel_df['speed'].mean(), color='red', linestyle='--', label=f'Mean: {vessel_df["speed"].mean():.1f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Vessel type distribution
ax3 = axes[1, 0]
vessel_type_counts = vessel_df['vessel_type'].value_counts()
ax3.pie(vessel_type_counts.values, labels=vessel_type_counts.index, autopct='%1.1f%%')
ax3.set_title('Vessel Type Distribution')

# 4. Activity by time of day
ax4 = axes[1, 1]
ax4.hist(vessel_df['time_of_day'], bins=24, alpha=0.7, edgecolor='black')
ax4.set_xlabel('Hour of Day')
ax4.set_ylabel('Activity Count')
ax4.set_title('Vessel Activity by Time of Day')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nData Summary:")
print(f"- Latitude range: {vessel_df['latitude'].min():.2f} to {vessel_df['latitude'].max():.2f}")
print(f"- Longitude range: {vessel_df['longitude'].min():.2f} to {vessel_df['longitude'].max():.2f}")
print(f"- Speed range: {vessel_df['speed'].min():.1f} to {vessel_df['speed'].max():.1f} km/h")
print(f"- Vessels in Arctic waters: {len(vessel_df[vessel_df['in_arctic']])}")

## 3. Pattern Detection

Now let's analyze vessel behavior patterns to identify interesting activities.

In [ ]:
# Initialize pattern detector
pattern_detector = PatternDetector()

# Detect loitering behavior
print("Detecting loitering behavior...")
loitering_incidents = pattern_detector.detect_loitering(
    vessel_df, 
    speed_threshold=2.0, 
    duration_threshold_hours=1.5
)

# Detect unusual speeds
print("Detecting unusual speed patterns...")
unusual_speeds = pattern_detector.detect_unusual_speeds(vessel_df)

# Detect night activity
print("Detecting night-time activity...")
night_activity = pattern_detector.detect_night_activity(vessel_df)

# Display results
print(f"\n=== PATTERN DETECTION RESULTS ===")
print(f"Loitering incidents: {len(loitering_incidents)}")
print(f"Unusual speed incidents: {len(unusual_speeds)}")
print(f"Night activity incidents: {len(night_activity)}")

# Show details of interesting patterns
if loitering_incidents:
    print(f"\nLoitering Incidents:")
    for i, incident in enumerate(loitering_incidents[:3]):  # Show first 3
        print(f"  {i+1}. Vessel {incident['vessel_id']}: {incident['duration_hours']:.1f} hours")
        print(f"     Position: ({incident['average_position'][0]:.3f}, {incident['average_position'][1]:.3f})")
        print(f"     Average speed: {incident['average_speed']:.1f} km/h")

if unusual_speeds:
    print(f"\nUnusual Speed Incidents:")
    for i, incident in enumerate(unusual_speeds[:3]):  # Show first 3
        print(f"  {i+1}. Vessel {incident['vessel_id']}: {incident['speed']:.1f} km/h ({incident['severity']})")
        print(f"     Threshold: {incident['threshold']:.1f} km/h")

## 4. Fleet Pattern Analysis

Let's analyze patterns across the entire fleet to understand collective behavior.

In [ ]:
# Import fleet analysis module
from analysis.simple_patterns import FleetAnalyzer

# Initialize fleet analyzer
fleet_analyzer = FleetAnalyzer()

# Analyze fleet behavior
print("Analyzing fleet behavior patterns...")
fleet_analysis = fleet_analyzer.analyze_fleet_behavior(vessel_df)

# Display fleet statistics
print(f"\n=== FLEET ANALYSIS RESULTS ===")
print(f"Fleet size: {fleet_analysis['fleet_size']} vessels")
print(f"Analysis period: {fleet_analysis['analysis_period']['start']} to {fleet_analysis['analysis_period']['end']}")

# Fleet statistics
stats = fleet_analysis['fleet_statistics']
print(f"\nFleet Statistics:")
print(f"- Average speed: {stats['average_speed']['mean']:.1f} ± {stats['average_speed']['std']:.1f} km/h")
print(f"- Total distance: {stats['total_distance']['total']:.1f} km")
print(f"- Average track duration: {stats['track_duration']['mean']:.1f} hours")

print(f"\nBehavior Classification:")
for behavior, count in stats['behavior_distribution'].items():
    percentage = (count / stats['vessels_analyzed']) * 100
    print(f"- {behavior}: {count} vessels ({percentage:.1f}%)")

# Fleet patterns
patterns = fleet_analysis['fleet_patterns']
print(f"\nFleet Patterns:")
print(f"- Coordinated movements: {len(patterns['coordinated_movement'])}")
print(f"- Potential rendezvous points: {len(patterns['rendezvous_points'])}")

## 5. Anomaly Detection with Autoencoders

Let's train an autoencoder to detect anomalous vessel behavior patterns.

In [ ]:
# Import simple autoencoder module
from models.simple_autoencoder import SimpleAnomalyDetector

# Prepare data for autoencoder
print("Preparing data for anomaly detection...")

# Add required features for autoencoder
vessel_df_ml = vessel_df.copy()
vessel_df_ml['distance_to_shore'] = vessel_df_ml['distance_to_shore'].clip(0, 500)  # Cap at 500km
vessel_df_ml = vessel_df_ml.fillna(0)

print(f"Training data shape: {vessel_df_ml.shape}")
print(f"Features for ML: {['latitude', 'longitude', 'speed', 'heading', 'distance_to_shore', 'time_of_day', 'day_of_week', 'vessel_length']}")

# Initialize anomaly detector
anomaly_detector = SimpleAnomalyDetector(input_features=8)

# Train the autoencoder
print("\nTraining autoencoder for anomaly detection...")
training_history = anomaly_detector.train(vessel_df_ml, epochs=30, validation_split=0.2)

print(f"Training completed!")
print(f"Final training loss: {training_history['loss'][-1]:.4f}")
print(f"Final validation loss: {training_history['val_loss'][-1]:.4f}")
print(f"Anomaly threshold: {training_history['threshold']:.4f}")

In [ ]:
# Visualize training progress
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training loss
ax1 = axes[0]
ax1.plot(training_history['loss'], label='Training Loss', color='blue')
ax1.plot(training_history['val_loss'], label='Validation Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Autoencoder Training Progress')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Detect anomalies
print("Detecting anomalies in vessel behavior...")
anomaly_results = anomaly_detector.predict_anomaly(vessel_df_ml)

# Extract anomaly scores
anomaly_scores = [result['anomaly_score'] for result in anomaly_results]
is_anomaly = [result['is_anomaly'] for result in anomaly_results]

# Anomaly score distribution
ax2 = axes[1]
ax2.hist(anomaly_scores, bins=20, alpha=0.7, edgecolor='black')
ax2.axvline(training_history['threshold'], color='red', linestyle='--', label=f'Threshold: {training_history["threshold"]:.3f}')
ax2.set_xlabel('Anomaly Score')
ax2.set_ylabel('Count')
ax2.set_title('Anomaly Score Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Anomaly summary
n_anomalies = sum(is_anomaly)
anomaly_rate = (n_anomalies / len(anomaly_results)) * 100

print(f"\n=== ANOMALY DETECTION RESULTS ===")
print(f"Total vessels analyzed: {len(anomaly_results)}")
print(f"Anomalous vessels detected: {n_anomalies} ({anomaly_rate:.1f}%)")
print(f"Average anomaly score: {np.mean(anomaly_scores):.3f}")
print(f"Maximum anomaly score: {np.max(anomaly_scores):.3f}")

In [ ]:
# Show top anomalies
print("Top 5 most anomalous vessels:")
anomaly_df = pd.DataFrame(anomaly_results)
anomaly_df['vessel_id'] = vessel_df['vessel_id'].values
top_anomalies = anomaly_df.nlargest(5, 'anomaly_score')

for i, (_, anomaly) in enumerate(top_anomalies.iterrows()):
    vessel_info = vessel_df[vessel_df['vessel_id'] == anomaly['vessel_id']].iloc[0]
    print(f"  {i+1}. Vessel {anomaly['vessel_id']}:")
    print(f"     Anomaly score: {anomaly['anomaly_score']:.3f}")
    print(f"     Speed: {vessel_info['speed']:.1f} km/h")
    print(f"     Type: {vessel_info['vessel_type']}")
    print(f"     Position: ({vessel_info['latitude']:.3f}, {vessel_info['longitude']:.3f})")
    print()

## 6. Risk Scoring Analysis

Finally, let's apply our risk scoring system to assess vessel threat levels.

In [ ]:
# Import risk scoring module
from analysis.simple_risk_scoring import SimpleRiskScorer

# Initialize risk scorer
risk_scorer = SimpleRiskScorer()

# Convert vessel dataframe to risk scoring format
print("Preparing data for risk scoring...")
risk_data = []

for _, vessel in vessel_df.iterrows():
    # Get anomaly score for this vessel
    vessel_anomaly = anomaly_df[anomaly_df['vessel_id'] == vessel['vessel_id']]
    anomaly_score = vessel_anomaly['anomaly_score'].iloc[0] if len(vessel_anomaly) > 0 else 0
    
    vessel_risk_data = {
        'vessel_id': vessel['vessel_id'],
        'latitude': vessel['latitude'],
        'longitude': vessel['longitude'],
        'speed': vessel['speed'],
        'speed_variance': vessel['speed'] * np.random.uniform(0.1, 0.3),  # Simulated variance
        'vessel_type': vessel['vessel_type'],
        'vessel_length': vessel['vessel_length'],
        'has_ais': True,  # All our simulated vessels have AIS
        'distance_to_shore': vessel['distance_to_shore'],
        'time_of_day': vessel['time_of_day'],
        'day_of_week': vessel['day_of_week'],
        'operation_duration_hours': np.random.exponential(8),  # Simulated
        'movement_pattern': 'regular' if anomaly_score < 0.5 else 'irregular',
        'anomaly_score': anomaly_score
    }
    risk_data.append(vessel_risk_data)

print(f"Prepared {len(risk_data)} vessels for risk assessment")

# Calculate risk scores
print("\nCalculating risk scores...")
fleet_risk_analysis = risk_scorer.score_vessel_fleet(risk_data)

print(f"Risk assessment completed!")

In [ ]:
# Display risk analysis results
print(f"=== FLEET RISK ANALYSIS ===")
print(f"Fleet size: {fleet_risk_analysis['fleet_size']} vessels")
print(f"Average risk score: {fleet_risk_analysis['average_risk_score']}/10")
print(f"Fleet risk level: {fleet_risk_analysis['fleet_risk_level']}")

print(f"\nRisk Level Distribution:")
for level, count in fleet_risk_analysis['risk_level_distribution'].items():
    percentage = (count / fleet_risk_analysis['fleet_size']) * 100
    print(f"- {level}: {count} vessels ({percentage:.1f}%)")

print(f"\nTop 5 Highest Risk Vessels:")
for i, vessel in enumerate(fleet_risk_analysis['highest_risk_vessels']):
    print(f"  {i+1}. {vessel['vessel_id']}: {vessel['total_risk_score']}/10 ({vessel['risk_level']})")
    if vessel['risk_factors']:
        print(f"     Risk factors: {', '.join(vessel['risk_factors'][:3])}")
    print()

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Extract risk scores
risk_scores = [vessel['total_risk_score'] for vessel in fleet_risk_analysis['all_vessels']]
risk_levels = [vessel['risk_level'] for vessel in fleet_risk_analysis['all_vessels']]

# 1. Risk score distribution
ax1 = axes[0, 0]
ax1.hist(risk_scores, bins=15, alpha=0.7, edgecolor='black')
ax1.axvline(fleet_risk_analysis['average_risk_score'], color='red', linestyle='--', 
           label=f'Average: {fleet_risk_analysis["average_risk_score"]}')
ax1.set_xlabel('Risk Score')
ax1.set_ylabel('Count')
ax1.set_title('Risk Score Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Risk level pie chart
ax2 = axes[0, 1]
risk_counts = list(fleet_risk_analysis['risk_level_distribution'].values())
risk_labels = list(fleet_risk_analysis['risk_level_distribution'].keys())
colors = ['green', 'yellow', 'orange', 'red', 'darkred']
ax2.pie(risk_counts, labels=risk_labels, autopct='%1.1f%%', colors=colors[:len(risk_labels)])
ax2.set_title('Risk Level Distribution')

# 3. Risk vs Anomaly Score
ax3 = axes[1, 0]
vessel_anomaly_scores = [vessel['anomaly_score'] for vessel in risk_data]
ax3.scatter(vessel_anomaly_scores, risk_scores, alpha=0.6)
ax3.set_xlabel('Anomaly Score')
ax3.set_ylabel('Risk Score')
ax3.set_title('Risk Score vs Anomaly Score')
ax3.grid(True, alpha=0.3)

# Add correlation coefficient
correlation = np.corrcoef(vessel_anomaly_scores, risk_scores)[0, 1]
ax3.text(0.05, 0.95, f'Correlation: {correlation:.3f}', transform=ax3.transAxes, 
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 4. Risk by vessel type
ax4 = axes[1, 1]
risk_by_type = {}
for vessel in fleet_risk_analysis['all_vessels']:
    vessel_data = next(v for v in risk_data if v['vessel_id'] == vessel['vessel_id'])
    vessel_type = vessel_data['vessel_type']
    if vessel_type not in risk_by_type:
        risk_by_type[vessel_type] = []
    risk_by_type[vessel_type].append(vessel['total_risk_score'])

types = list(risk_by_type.keys())
avg_risks = [np.mean(risk_by_type[t]) for t in types]
ax4.bar(types, avg_risks, alpha=0.7)
ax4.set_xlabel('Vessel Type')
ax4.set_ylabel('Average Risk Score')
ax4.set_title('Average Risk by Vessel Type')
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary and Conclusions

Let's summarize our findings from this Arctic vessel analysis.

In [ ]:
# Create comprehensive summary
print("=" * 60)
print("ARCTIC VESSEL ANALYSIS SUMMARY")
print("=" * 60)

print(f"\n📊 DATA OVERVIEW:")
print(f"• Total vessel positions analyzed: {len(vessel_df)}")
print(f"• Unique vessels tracked: {vessel_df['vessel_id'].nunique()}")
print(f"• Time period: {(vessel_df['timestamp'].max() - vessel_df['timestamp'].min()).total_seconds() / 3600:.1f} hours")
print(f"• Geographic area: {vessel_df['latitude'].min():.2f}°N to {vessel_df['latitude'].max():.2f}°N")

print(f"\n🔍 PATTERN DETECTION:")
print(f"• Loitering incidents: {len(loitering_incidents)}")
print(f"• Unusual speed events: {len(unusual_speeds)}")
print(f"• Night activity incidents: {len(night_activity)}")
print(f"• Coordinated movements: {len(patterns['coordinated_movement'])}")

print(f"\n🤖 ANOMALY DETECTION:")
print(f"• Model training epochs: 30")
print(f"• Anomaly threshold: {training_history['threshold']:.4f}")
print(f"• Vessels flagged as anomalous: {n_anomalies} ({anomaly_rate:.1f}%)")
print(f"• Average anomaly score: {np.mean(anomaly_scores):.3f}")

print(f"\n⚠️ RISK ASSESSMENT:")
print(f"• Fleet average risk: {fleet_risk_analysis['average_risk_score']}/10")
print(f"• Fleet risk level: {fleet_risk_analysis['fleet_risk_level']}")
print(f"• High/Critical risk vessels: {fleet_risk_analysis['risk_level_distribution'].get('HIGH', 0) + fleet_risk_analysis['risk_level_distribution'].get('CRITICAL', 0)}")

print(f"\n🎓 EDUCATIONAL INSIGHTS:")
print(f"• Demonstrated vessel tracking and pattern recognition")
print(f"• Applied machine learning for anomaly detection")
print(f"• Implemented multi-factor risk scoring methodology")
print(f"• Showed correlation between anomaly and risk scores: {correlation:.3f}")

print(f"\n📈 DATA SCIENCE TECHNIQUES DEMONSTRATED:")
print(f"• Geospatial data processing and visualization")
print(f"• Time series analysis of vessel movements")
print(f"• Unsupervised learning with autoencoders")
print(f"• Multi-criteria decision analysis for risk scoring")
print(f"• Statistical pattern recognition")

print(f"\n✅ ANALYSIS COMPLETED SUCCESSFULLY!")
print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

## Next Steps for Further Research

This notebook demonstrated the core functionality of the ArcticShadowTracker system. For further educational exploration, consider:

1. **Real Data Integration**: Replace simulated data with actual AIS feeds from maritime authorities
2. **Satellite Imagery**: Add satellite image processing for vessel detection
3. **Advanced ML Models**: Experiment with different neural network architectures
4. **Temporal Analysis**: Implement time series forecasting for vessel behavior prediction
5. **Interactive Visualization**: Create web-based dashboards for real-time monitoring

**Academic Note**: This project serves as an educational foundation for understanding data science applications in maritime research and Arctic studies.